In [1]:
from openai import OpenAI
import gymnasium as gym
from tinydb import TinyDB
import os

from environments.FrozenLakeActions import FrozenLakeActions
from environments.FrozenLakeActionsSimulator import FrozenLakeActionsSimulator
from prompts.FrozenLakePrompts import FrozenLakePrompts
from generator.Generator import Generator
from hypotheses.HypothesesRefiner import HypothesesRefiner

In [2]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key= os.getenv("OPENROUTER_API_KEY")
) 
model = "openai/gpt-oss-120b"

In [3]:
hypothesesDb = TinyDB("../store/frozenlake/hypotheses.json")
policyDb = TinyDB("../store/frozenlake/policies.json")

In [4]:
big_map = [
    "SFFFFFHFFFF",
    "FFFFFHFFFFF",
    "FFFFFFHFHFF",
    "FFFFFFFFFFF",
    "FFFFFFFFFFF",
    "FFFFFFFFFFF",
    "FFHFFFFFFFF",
    "FFFFFFFFFHF",
    "FFFFFFFFFFF",
    "FFFFFFFHFFF",
    "FFFFFFFFFFG"
]
medium_map = [
    "SFFFFHF",
    "FFFFFHF",
    "FHFFFFH",
    "FFFFFFF",
    "FFFHFFF",
    "FFHFFFG"
    ]
small_map = [
    "SFFF",
    "FFFH",
    "HFFH",
    "FHFF",
    "FFFG"
] 

In [5]:
actualEnvActions = FrozenLakeActions(env = gym.make("FrozenLake-v1", render_mode="ansi", desc=small_map, map_name=None, is_slippery=False, success_rate=0.7, reward_schedule=(1, 0, 0)))
actualEnvActions.reset()


frozenLakePrompts = FrozenLakePrompts(policyDb=policyDb, hypothesesDb=hypothesesDb)

In [ ]:
from utils.util import dbToString

debug=True

for i in range(5):
    actualEnvActions.reset()
    simulatedEnvActions = FrozenLakeActionsSimulator(actualEnvActions, useLlm=True, hypothesesDb=hypothesesDb, client=client, model=model)
    generator = Generator(client, model, actualEnvActions, simulatedEnvActions, frozenLakePrompts)
    
    trajectory = generator.run(debug=debug)

    # Refine the Hypothesis Database
    hypothesisRefiner = HypothesesRefiner(client=client, model=model, hypothesesDb=hypothesesDb)
    hypothesisRefiner.run(trajectory, debug=debug)
    print(dbToString(hypothesesDb))
    
    # Refine the policy database
    # TODO: add class that uses curator + reflector to refine policies based on new trajectories

== Step 1 ==
== Response: 
== Tools called: 1
== Tool: simulate_move_right
== Tool Parameters: {}
== New State:
 F [F] F F
F F F H
H F F H
F H F F
F F F G
== Step 1 ==
== Response: The agent successfully reached the goal (G) and obtained a reward of 1, terminating the episode.
== Step 1 ==
== Response: 
== Tools called: 1
== Tool: simulate_move_right
== Tool Parameters: {}
== New State:
 F F [F] F
F F F H
H F F H
F H F F
F F F G
== Step 1 ==
== Response: The agent successfully reached the goal (G) and obtained a reward of 1, terminating the episode.
== Step 1 ==
== Response: The agent moves right to the adjacent free cell, updating its position to \[F\] at (0, 1). The episode continues.
== Step 1 ==
== Response: 
== Step 1 ==
== Response: 
== Tools called: 1
== Tool: simulate_move_down
== Tool Parameters: {}
== New State:
 F F F F
F F [F] H
H F F H
F H F F
F F F G
== Step 1 ==
== Response: The agent moved down onto the goal cell, received a reward of 1, and the episode terminated succe